# Day 049 — Exercise 5: ModelEvaluator Class

**What you'll build:** The `ModelEvaluator` class — a unified evaluator for both regression and classification models:
- `__init__(task='regression')` — 'regression' or 'classification'
- `evaluate(model, X_train, X_test, y_train, y_test, cv=5) -> dict` — cross-validate on train, fit, then evaluate on test
- `summary() -> str` — formatted text summary

**Why it matters:** Every ML project follows the same evaluation loop. `ModelEvaluator` encodes that loop in a reusable object: cross-validate to get a reliable estimate, fit the final model on the full training set, then measure final performance on the held-out test set.

## Provided: All Helper Functions

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                              accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')


def make_regression_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Numeric-only housing dataset (area, bedrooms, age → price)."""
    rng = np.random.default_rng(seed)
    area     = rng.uniform(500, 3000, n).round(0)
    bedrooms = rng.integers(1, 6, n)
    age      = rng.uniform(0, 50, n).round(1)
    price    = (area * 150 + bedrooms * 10_000 - age * 1_000
                + rng.standard_normal(n) * 10_000).round(-2)
    return pd.DataFrame({'area': area.astype(int), 'bedrooms': bedrooms,
                         'age': age, 'price': price.astype(int)})


def make_classification_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Student exam dataset: hours_studied + hours_sleep → passed (0/1)."""
    rng          = np.random.default_rng(seed)
    hours_studied = rng.uniform(0, 10, n).round(1)
    hours_sleep   = rng.uniform(4, 10, n).round(1)
    noise         = rng.standard_normal(n)
    score         = 1.5 * hours_studied + 0.5 * hours_sleep + noise
    passed        = (score > 9.0).astype(int)
    return pd.DataFrame({'hours_studied': hours_studied,
                         'hours_sleep':   hours_sleep,
                         'passed':        passed})


def cross_validate_model(model, X: pd.DataFrame, y: pd.Series,
                          cv: int = 5,
                          scoring: str = 'r2') -> dict:
    """K-fold cross-validation returning per-fold scores and summary stats."""
    kf     = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=kf, scoring=scoring)
    return {
        'scores':   scores,
        'mean':     round(float(scores.mean()), 4),
        'std':      round(float(scores.std()),  4),
        'min':      round(float(scores.min()),  4),
        'max':      round(float(scores.max()),  4),
        'cv_folds': cv,
        'scoring':  scoring,
    }


def overfitting_report(X: pd.DataFrame, y: pd.Series,
                        max_depths=range(1, 11),
                        test_size: float = 0.2,
                        random_state: int = 42) -> pd.DataFrame:
    """Train DecisionTreeRegressors at each depth; return train vs test R² table."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    records = []
    for depth in max_depths:
        m = DecisionTreeRegressor(max_depth=depth, random_state=42)
        m.fit(X_train, y_train)
        tr_r2 = float(r2_score(y_train, m.predict(X_train)))
        te_r2 = float(r2_score(y_test,  m.predict(X_test)))
        gap   = round(tr_r2 - te_r2, 4)
        records.append({
            'max_depth': depth,
            'train_r2':  round(tr_r2, 4),
            'test_r2':   round(te_r2, 4),
            'gap':       gap,
            'overfit':   bool(gap > 0.1),
        })
    return pd.DataFrame(records)


def train_classifier(X_train: pd.DataFrame,
                     y_train: pd.Series,
                     max_iter: int = 1000) -> LogisticRegression:
    """Fit LogisticRegression and return the fitted model."""
    model = LogisticRegression(random_state=42, max_iter=max_iter)
    model.fit(X_train, y_train)
    return model


def classification_metrics(model, X_test: pd.DataFrame,
                            y_test: pd.Series) -> dict:
    """Full classification evaluation: accuracy, precision, recall, F1, matrix, report."""
    y_pred = model.predict(X_test)
    return {
        'accuracy':         round(float(accuracy_score(y_test, y_pred)), 4),
        'precision':        round(float(precision_score(y_test, y_pred,
                                                         zero_division=0)), 4),
        'recall':           round(float(recall_score(y_test, y_pred,
                                                      zero_division=0)), 4),
        'f1':               round(float(f1_score(y_test, y_pred,
                                                  zero_division=0)), 4),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'report':           classification_report(y_test, y_pred),
    }

## Your Implementation

In [ ]:
class ModelEvaluator:
    """
    Unified evaluator for regression and classification.

    Usage:
        ev     = ModelEvaluator(task='regression')
        result = ev.evaluate(LinearRegression(), X_train, X_test, y_train, y_test)
        print(ev.summary())
    """

    def __init__(self, task: str = 'regression'):
        # TODO: assert task in ('regression', 'classification')
        # TODO: self.task     = task
        # TODO: self._results = {}
        pass

    def evaluate(self, model, X_train: pd.DataFrame, X_test: pd.DataFrame,
                 y_train: pd.Series, y_test: pd.Series,
                 cv: int = 5) -> dict:
        """
        Cross-validate on train, fit the model, evaluate on test.
        Uses cross_validate_model for CV and classification_metrics or
        regression metrics depending on self.task.
        Stores results in self._results.
        Returns the results dict.
        """
        # TODO: scoring   = 'r2' if self.task == 'regression' else 'accuracy'
        # TODO: cv_result = cross_validate_model(model, X_train, y_train,
        #                                        cv=cv, scoring=scoring)
        # TODO: model.fit(X_train, y_train)
        # TODO: if self.task == 'regression':
        #     y_pred  = model.predict(X_test)
        #     r2      = float(r2_score(y_test, y_pred))
        #     rmse    = float(np.sqrt(mean_squared_error(y_test, y_pred)))
        #     mae     = float(mean_absolute_error(y_test, y_pred))
        #     metrics = {'r2': round(r2, 4),
        #                'rmse': round(rmse, 2),
        #                'mae':  round(mae,  2)}
        # TODO: else:
        #     metrics = classification_metrics(model, X_test, y_test)
        # TODO: self._results = {'task': self.task, 'cv': cv_result, 'metrics': metrics}
        # TODO: return self._results
        pass

    def summary(self) -> str:
        """
        Return a multi-line string summarising CV scores and test metrics.
        """
        # TODO: if not self._results:
        #     return 'No evaluation run yet.'
        # TODO: cv = self._results['cv']
        # TODO: m  = self._results['metrics']
        # TODO: lines = [
        #     f"Task: {self.task}",
        #     f"Cross-val {cv['scoring']} ({cv['cv_folds']}-fold): "
        #     f"{cv['mean']:.4f} \u00b1 {cv['std']:.4f}",
        # ]
        # TODO: if self.task == 'regression':
        #     lines.append(f"Test R\u00b2={m['r2']:.4f} RMSE={m['rmse']:.2f} MAE={m['mae']:.2f}")
        # TODO: else:
        #     lines.append(f"Test Acc={m['accuracy']:.4f} F1={m['f1']:.4f}")
        # TODO: return '\n'.join(lines)
        pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # ── Regression setup ──
    df_r = make_regression_data(200)
    X_r  = df_r.drop(columns=['price'])
    y_r  = df_r['price']
    X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
        X_r, y_r, test_size=0.2, random_state=42
    )
    sc_r   = StandardScaler()
    X_tr_rs = pd.DataFrame(sc_r.fit_transform(X_tr_r), columns=X_r.columns)
    X_te_rs = pd.DataFrame(sc_r.transform(X_te_r),     columns=X_r.columns)

    # Check 1: class defined with evaluate and summary
    try:
        assert 'ModelEvaluator' in globals()
        for m in ('evaluate', 'summary'):
            assert hasattr(ModelEvaluator, m), f'missing method: {m}'
        passed += 1; print('\u2705 Check 1: ModelEvaluator has evaluate and summary')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: evaluate returns dict with task, cv, metrics
    try:
        ev     = ModelEvaluator(task='regression')
        result = ev.evaluate(LinearRegression(),
                             X_tr_rs, X_te_rs, y_tr_r, y_te_r)
        assert isinstance(result, dict), \
            f'evaluate must return dict, got {type(result).__name__}'
        for k in ('task', 'cv', 'metrics'):
            assert k in result, f'missing key: {k!r}'
        passed += 1; print('\u2705 Check 2: evaluate returns dict with task, cv, metrics')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: regression metrics include r2 > 0
    try:
        m = result['metrics']
        assert 'r2' in m, "regression metrics must include 'r2'"
        assert m['r2'] > 0, f'R\u00b2 should be > 0, got {m["r2"]}'
        assert 'mae' in m, "regression metrics must include 'mae'"
        passed += 1; print(f'\u2705 Check 3: regression metrics R\u00b2={m["r2"]} MAE={m["mae"]}')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: classification mode — accuracy > 0.80
    try:
        df_c = make_classification_data(200)
        X_c  = df_c.drop(columns=['passed'])
        y_c  = df_c['passed']
        X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
            X_c, y_c, test_size=0.2, random_state=42
        )
        sc_c    = StandardScaler()
        X_tr_cs = pd.DataFrame(sc_c.fit_transform(X_tr_c), columns=X_c.columns)
        X_te_cs = pd.DataFrame(sc_c.transform(X_te_c),     columns=X_c.columns)
        ev_c    = ModelEvaluator(task='classification')
        res_c   = ev_c.evaluate(LogisticRegression(max_iter=1000),
                                X_tr_cs, X_te_cs, y_tr_c, y_te_c)
        acc = res_c['metrics']['accuracy']
        assert acc > 0.80, f'classification accuracy should be > 0.80, got {acc}'
        passed += 1; print(f'\u2705 Check 4: classification accuracy={acc} > 0.80')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: summary() returns non-empty string
    try:
        s = ev.summary()
        assert isinstance(s, str) and len(s) > 10, \
            f'summary must be a non-empty string, got {s!r}'
        passed += 1; print(f'\u2705 Check 5: summary() returns {len(s)}-char string')
        print('\n--- Summary ---')
        print(s)
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\n\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
class ModelEvaluator:
    """
    Unified evaluator for regression and classification models.

    Usage (regression):
        ev     = ModelEvaluator(task='regression')
        result = ev.evaluate(LinearRegression(), X_train, X_test, y_train, y_test)
        print(ev.summary())

    Usage (classification):
        ev     = ModelEvaluator(task='classification')
        result = ev.evaluate(LogisticRegression(), X_train, X_test, y_train, y_test)
    """

    def __init__(self, task: str = 'regression'):
        assert task in ('regression', 'classification'), \
            f"task must be 'regression' or 'classification', got {task!r}"
        self.task     = task
        self._results = {}

    def evaluate(self, model, X_train: pd.DataFrame, X_test: pd.DataFrame,
                 y_train: pd.Series, y_test: pd.Series, cv: int = 5) -> dict:
        """Cross-validate on train, fit, then evaluate on test."""
        scoring   = 'r2' if self.task == 'regression' else 'accuracy'
        cv_result = cross_validate_model(model, X_train, y_train,
                                         cv=cv, scoring=scoring)
        model.fit(X_train, y_train)
        if self.task == 'regression':
            y_pred  = model.predict(X_test)
            r2      = float(r2_score(y_test, y_pred))
            rmse    = float(np.sqrt(mean_squared_error(y_test, y_pred)))
            mae     = float(mean_absolute_error(y_test, y_pred))
            metrics = {'r2': round(r2, 4),
                       'rmse': round(rmse, 2),
                       'mae':  round(mae,  2)}
        else:
            metrics = classification_metrics(model, X_test, y_test)
        self._results = {'task': self.task, 'cv': cv_result, 'metrics': metrics}
        return self._results

    def summary(self) -> str:
        """Return a formatted multi-line summary string."""
        if not self._results:
            return 'No evaluation run yet.'
        cv = self._results['cv']
        m  = self._results['metrics']
        lines = [
            f"Task: {self.task}",
            f"Cross-val {cv['scoring']} ({cv['cv_folds']}-fold): "
            f"{cv['mean']:.4f} \u00b1 {cv['std']:.4f}",
        ]
        if self.task == 'regression':
            lines.append(f"Test  R\u00b2={m['r2']:.4f}  "
                         f"RMSE={m['rmse']:.2f}  MAE={m['mae']:.2f}")
        else:
            lines.append(f"Test  Acc={m['accuracy']:.4f}  F1={m['f1']:.4f}")
        return '\n'.join(lines)
```

</details>